# **We have got the dataset from the specific location and now we are good to go with our further operations on dataset:**

# Data Validation in Fraud Detection — Theoretical & Intuitive View

| Validation Step | What We Are Really Asking | Why It Matters in a Real Fraud System | Intuitive / Business Meaning | What Can Go Wrong If We Skip It |
|-----------------|---------------------------|---------------------------------------|------------------------------|---------------------------------|
| **1. Target Distribution (Fraud Rate)** | How rare is fraud in this population? | Fraud is almost always a minority class. The rarity directly determines which metrics are meaningful, how we should set thresholds, and whether class imbalance techniques are required. | A model that always says “not fraud” can look extremely accurate while being completely useless. We need to know the base rate so we can judge real detection power versus customer friction. | We optimise for accuracy, celebrate high numbers, and deploy a model that blocks almost no fraud (or blocks far too many legitimate customers). |
| **2. Missingness Landscape** | Is missingness random, or does it carry behavioural / channel meaning? | In payment systems, many fields are systematically missing for certain product types, regions, devices, or attack patterns. Missingness itself is often a signal. | A transaction with no device fingerprint may be harder for an attacker to hide — or easier. Knowing the pattern tells us whether “missing” should later become a feature. | We blindly impute or drop columns, destroy a useful signal, or create features that only exist for a biased subset of traffic. |
| **3. Constant / Near-Constant Columns** | Does this column ever change? | A feature that never varies cannot help separate fraud from legitimate behaviour. It also often indicates a deprecated field, logging error, or extremely narrow segment. | Keeping constant columns wastes memory, confuses feature importance, and can make pipelines slower for zero gain. | We carry noise into modelling, inflate dimensionality, and later wonder why certain features have zero importance. |
| **4. Cardinality of Categorical Variables** | How many distinct values does this field take? | High-cardinality fields (card IDs, email domains, DeviceInfo, etc.) can be extremely powerful identity signals, but they also create severe risks of overfitting, memory explosion, and entity leakage. | The same card or device appearing repeatedly is often a strong behavioural clue. But treating every unique value as a separate category can make the model memorise training entities instead of learning general patterns. | We one-hot encode recklessly, create millions of sparse columns, leak future identity information, or discard a strong signal because it looked “too messy.” |
| **5. Relationship Between Transaction & Identity Tables** | What fraction of transactions actually have richer identity / device context? | In real payment flows, identity data is frequently missing. A production system must score both fully observed and sparsely observed transactions. | Only ~24% of transactions in this dataset have identity information. That is not a defect — it is a structural property of the ecosystem. The model must remain useful for the other 76%. | We build features that only work when identity data is present, creating a model that is effectively blind for the majority of live traffic. |
| **6. Suspicious / Impossible Values** | Are there values that violate basic domain constraints? | Some anomalies are pure data-quality bugs. Others are genuine rare fraud patterns that look “impossible” under normal assumptions. | Negative amounts, future timestamps, or extreme outliers can either be pipeline errors or early warning signs of sophisticated attacks. | We either train on corrupted data or accidentally remove rare but highly informative fraud cases. |

---

**Core Principle**

Data validation in fraud detection is not a cleaning ritual.  
It is the first rigorous attempt to understand how the observable transaction world relates to the hidden fraud world — how rare the event is, how incomplete our visibility is, which signals are stable, and which ones are fragile.

Only after this understanding is in place do we earn the right to engineer features or train models.

In [1]:
# Libraries:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Training dataset
train = pd.read_pickle(r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\train.pkl')

In [3]:
train.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [4]:
# Test dataset:
test = pd.read_pickle(r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\test.pkl')

In [5]:
test.head()

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.00,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.00,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.95,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.95,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), str(31)
memory usage: 1.9 GB


In [7]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 506691 entries, 0 to 506690
Columns: 433 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(3), str(31)
memory usage: 1.6 GB


# Let's just go with or dataset and see the chracteristics of our whole dataset.

In [8]:
## Train dataset:
# Checking for missing values:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), str(31)
memory usage: 1.9 GB


In [9]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 506691 entries, 0 to 506690
Columns: 433 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(3), str(31)
memory usage: 1.6 GB


In [10]:
train.columns.to_list()

['TransactionID',
 'isFraud',
 'TransactionDT',
 'TransactionAmt',
 'ProductCD',
 'card1',
 'card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'dist1',
 'dist2',
 'P_emaildomain',
 'R_emaildomain',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'C7',
 'C8',
 'C9',
 'C10',
 'C11',
 'C12',
 'C13',
 'C14',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'D13',
 'D14',
 'D15',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'V29',
 'V30',
 'V31',
 'V32',
 'V33',
 'V34',
 'V35',
 'V36',
 'V37',
 'V38',
 'V39',
 'V40',
 'V41',
 'V42',
 'V43',
 'V44',
 'V45',
 'V46',
 'V47',
 'V48',
 'V49',
 'V50',
 'V51',
 'V52',
 'V53',
 'V54',
 'V55',
 'V56',
 'V57',
 'V58',
 'V59',
 'V60',
 'V61',
 'V62',
 'V63',
 'V64',
 'V

In [11]:
test.columns.to_list()

['TransactionID',
 'TransactionDT',
 'TransactionAmt',
 'ProductCD',
 'card1',
 'card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'dist1',
 'dist2',
 'P_emaildomain',
 'R_emaildomain',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'C7',
 'C8',
 'C9',
 'C10',
 'C11',
 'C12',
 'C13',
 'C14',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'D13',
 'D14',
 'D15',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'V29',
 'V30',
 'V31',
 'V32',
 'V33',
 'V34',
 'V35',
 'V36',
 'V37',
 'V38',
 'V39',
 'V40',
 'V41',
 'V42',
 'V43',
 'V44',
 'V45',
 'V46',
 'V47',
 'V48',
 'V49',
 'V50',
 'V51',
 'V52',
 'V53',
 'V54',
 'V55',
 'V56',
 'V57',
 'V58',
 'V59',
 'V60',
 'V61',
 'V62',
 'V63',
 'V64',
 'V65',
 'V66',

In [12]:
# Checking for missing values:
missing_cols_train = train.columns[train.isnull().any()]
missing_cols_train.to_list()

['card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'dist1',
 'dist2',
 'P_emaildomain',
 'R_emaildomain',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'D13',
 'D14',
 'D15',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'V29',
 'V30',
 'V31',
 'V32',
 'V33',
 'V34',
 'V35',
 'V36',
 'V37',
 'V38',
 'V39',
 'V40',
 'V41',
 'V42',
 'V43',
 'V44',
 'V45',
 'V46',
 'V47',
 'V48',
 'V49',
 'V50',
 'V51',
 'V52',
 'V53',
 'V54',
 'V55',
 'V56',
 'V57',
 'V58',
 'V59',
 'V60',
 'V61',
 'V62',
 'V63',
 'V64',
 'V65',
 'V66',
 'V67',
 'V68',
 'V69',
 'V70',
 'V71',
 'V72',
 'V73',
 'V74',
 'V75',
 'V76',
 'V77',
 'V78',
 'V79',
 'V80',
 'V81',
 'V82',
 'V83',
 'V84',
 'V85',
 'V86',
 'V87',
 'V88',
 'V89

In [13]:
# Checking for missing values:
missing_cols_test = test.columns[test.isnull().any()]
missing_cols_test.to_list()

['card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'dist1',
 'dist2',
 'P_emaildomain',
 'R_emaildomain',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'C7',
 'C8',
 'C9',
 'C10',
 'C11',
 'C12',
 'C13',
 'C14',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'D13',
 'D14',
 'D15',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'V29',
 'V30',
 'V31',
 'V32',
 'V33',
 'V34',
 'V35',
 'V36',
 'V37',
 'V38',
 'V39',
 'V40',
 'V41',
 'V42',
 'V43',
 'V44',
 'V45',
 'V46',
 'V47',
 'V48',
 'V49',
 'V50',
 'V51',
 'V52',
 'V53',
 'V54',
 'V55',
 'V56',
 'V57',
 'V58',
 'V59',
 'V60',
 'V61',
 'V62',
 'V63',
 'V64',
 'V65',
 'V66',
 'V67',
 'V68',
 'V69',
 'V70',
 'V71',
 'V72',
 'V73',
 'V74',
 'V75',
 'V76'

In [14]:
train[missing_cols_train].isnull().sum()

card2           8933
card3           1565
card4           1577
card5           4259
card6           1571
               ...  
id_36         449555
id_37         449555
id_38         449555
DeviceType    449730
DeviceInfo    471874
Length: 414, dtype: int64

In [15]:
test[missing_cols_test].isnull().sum()

card2           8654
card3           3002
card4           3086
card5           4547
card6           3007
               ...  
id-36         369714
id-37         369714
id-38         369714
DeviceType    369760
DeviceInfo    391634
Length: 385, dtype: int64

**Checking with some tweaks:**

Key questions we need to answer systematically:

* Target distribution
How rare is fraud? (This drives almost every later modelling decision.)

* Missingness landscape
Which columns have extreme missingness?

Is missingness random or patterned (especially with respect to isFraud)?

Difference between transaction columns and identity columns.

* Constant / near-constant columns
Columns that never change give no information and can safely be noted for later removal.

* Cardinality of categorical variables
Especially high-cardinality ones (card1, P_emaildomain, DeviceInfo, etc.).

* Relationship between the two tables
We already know only 24.4% of transactions have identity data. We should also confirm that every TransactionID in the identity table exists in the transaction table (no orphan identity rows).

* Suspicious values / data quality issues

Impossible amounts, negative values where they shouldn’t exist, etc.

In [24]:
train.columns.to_list()

['TransactionID',
 'isFraud',
 'TransactionDT',
 'TransactionAmt',
 'ProductCD',
 'card1',
 'card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'dist1',
 'dist2',
 'P_emaildomain',
 'R_emaildomain',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'C7',
 'C8',
 'C9',
 'C10',
 'C11',
 'C12',
 'C13',
 'C14',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'D13',
 'D14',
 'D15',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'V29',
 'V30',
 'V31',
 'V32',
 'V33',
 'V34',
 'V35',
 'V36',
 'V37',
 'V38',
 'V39',
 'V40',
 'V41',
 'V42',
 'V43',
 'V44',
 'V45',
 'V46',
 'V47',
 'V48',
 'V49',
 'V50',
 'V51',
 'V52',
 'V53',
 'V54',
 'V55',
 'V56',
 'V57',
 'V58',
 'V59',
 'V60',
 'V61',
 'V62',
 'V63',
 'V64',
 'V

In [ ]:
# Checking for the dev

print("Final train shape:", train.shape)
print("Number of transactions with any identity info:", 
      train['id_01'].notna().sum())          # or any other id_ column
print("Percentage with identity:", 
      train['id_01'].notna().mean() * 100)

Final train shape: (590540, 434)
Number of transactions with any identity info: 144233
Percentage with identity: 24.42391709283029


# **Only 24.4% of transactions have any identity/device information.**
This is normal in real payment systems and is itself an important signal.

In [20]:
print("Fraud rate:")
print(train['isFraud'].value_counts(normalize=True))
print("\nAbsolute counts:")
print(train['isFraud'].value_counts())

Fraud rate:
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64

Absolute counts:
isFraud
0    569877
1     20663
Name: count, dtype: int64


# **Till now, I think we are done with validation of our data which is combined all together for better processing.**

* Our data is aligned and merged perfectly.